# 08 — PCA dhe reduktimi dimensional

## Konteksti dhe problemi shkencor

Spektrat janë shembull natyror i të dhënave me shumë kolona të korreluara. PCA lidhet drejtpërdrejt me kovariancën, vlerat vetjake dhe SVD.

### Objektivat e të nxënit

- të implementoni PCA manualisht;
- të kuptoni explained variance dhe reconstruction;
- të interpretoni loadings;
- të dalloni kur scaling është i nevojshëm.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Spektra me shumë gjatësi vale

In [ ]:
df=pd.read_csv(DATA/'synthetic_spectra.csv'); X=df.to_numpy(); wavelengths=np.array([float(c.split('_')[1]) for c in df.columns]); plt.plot(wavelengths,X[:12].T,alpha=.6); plt.xlabel('λ'); plt.ylabel('I'); plt.show()

## 2. Qendrimi dhe kovarianca

In [ ]:
mean=X.mean(0); Xc=X-mean; C=np.cov(Xc,rowvar=False); print(C.shape, np.allclose(C,C.T))

## 3. PCA manuale me vlera vetjake

In [ ]:
vals,vecs=np.linalg.eigh(C); idx=np.argsort(vals)[::-1]; vals=vals[idx]; vecs=vecs[:,idx]; ratio=vals/vals.sum(); print(ratio[:8],ratio[:3].sum())

In [ ]:
Z=Xc@vecs[:,:3]; plt.scatter(Z[:,0],Z[:,1],s=14); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

## 4. Loadings si forma spektrale

In [ ]:
for j in range(3): plt.plot(wavelengths,vecs[:,j],label=f'PC{j+1}'); plt.legend(); plt.xlabel('λ'); plt.show()

## 5. Rindërtimi dhe gabimi

In [ ]:
errs=[]
for q in [1,2,3,5,10,20]:
 V=vecs[:,:q]; rec=(Xc@V)@V.T+mean; errs.append((q,np.mean((X-rec)**2)))
pd.DataFrame(errs,columns=['q','MSE_reconstruction'])

## 6. Krahasim me scikit-learn

In [ ]:
from sklearn.decomposition import PCA
p=PCA(n_components=5).fit(X); print(np.allclose(p.explained_variance_ratio_,ratio[:5])); print(np.abs(np.diag(p.components_@vecs[:,:5])))

Shenja e komponentit është arbitrare; për këtë arsye krahasojmë produktin skalar në vlerë absolute.

## 7. SVD direkt

In [ ]:
U,S,Vt=np.linalg.svd(Xc,full_matrices=False); print(np.allclose(S**2/(len(X)-1),vals))

## 8. Rast dështimi: shkallët e ndryshme

In [ ]:
Xmix=np.c_[Z[:,:2],10000*np.random.default_rng(1).normal(size=len(X))]; p1=PCA(2).fit(Xmix); p2=make_pipeline(StandardScaler(),PCA(2)).fit(Xmix); print('pa scaling',p1.explained_variance_ratio_)

## 9. Ushtrime

- Derivoni problemin e maksimizimit $v^TCv$.
- Zgjidhni q për 95% variancë.
- Krahasoni reconstruction error me explained variance.
- Interpretoni fizikisht maksimumet e tre loadings të para.

## 10. Shembull i plotë

Kompresoni 120 kanale në q komponentë, rindërtoni spektrat dhe raportoni raportin e kompresimit dhe gabimin.

## 11. Playground

Ndryshoni amplitudat e bazave spektrale në gjenerator dhe kontrolloni si ndryshon renditja e eigenvalues.